### Development version

Paths, input loading, and filtering compatibility were updated during
repository cleanup.

# NileTTS LLM-Generated QA Pairs + AraT5 Reference QA

1. Download NileTTS metadata only.
2. Generate QA pairs from **reference transcripts** using an LLM.
3. Validate/filter the generated QA pairs.
4. Fine-tune AraT5 on reference-transcript QA pairs.
5. Evaluate AraT5 on the official eval/test reference transcripts.


In [ ]:
# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "requirements-qa.txt").is_file():
        requirements_file = candidate / "requirements-qa.txt"
        break
else:
    raise FileNotFoundError("Could not find requirements-qa.txt")

%pip install -r "{requirements_file}"
# =====


In [2]:
import os
import re
import csv
import json
import time
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path

from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [ ]:
# =====
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (
        (candidate / "1_ASR").is_dir()
        and (candidate / "2_QA_GENERATION").is_dir()
    ):
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Open the notebook from inside the project folder."
    )

QA_DIR = PROJECT_ROOT / "2_QA_GENERATION"

TRAIN_FILE = (QA_DIR / "1_train_data" / "niletts_llm_qa_train.csv")
VAL_FILE = (QA_DIR / "2_val_data" / "niletts_llm_qa_val.csv")

CHECKPOINT_TEST_FILE = (QA_DIR / "3_test_data" / "checkpoints"/ "checkpoint_llm_qa_test.csv")
PROCESSED_TEST_FILE = (QA_DIR / "3_test_data" / "checkpoints"/ "processed_llm_qa_test.csv")

ASR_FILE = (PROJECT_ROOT / "1_ASR"/ "niletts_asr_eval_finetuning_transcriptions_metrics.csv")

CHECKPOINT_VAL_FILE = (QA_DIR / "2_val_data" / "checkpoints" / "checkpoint_llm_qa_val.csv")
PROCESSED_VAL_FILE = (QA_DIR / "2_val_data" / "checkpoints" / "processed_llm_qa_val.csv")
# =====

## 1. Download NileTTS metadata only

This QA notebook does not need audio files. It only needs the metadata files:

- `metadata_train.csv`: used to generate QA train/validation pairs.
- `metadata_eval.csv`: used to generate final QA test pairs aligned with the ASR eval split.


In [ ]:
# =====
# REPO_ID = "KickItLikeShika/NileTTS-dataset"

# train_meta_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename="metadata_train.csv")
# eval_meta_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename="metadata_eval.csv")
train_meta_path = QA_DIR / "0_data" / "metadata_train.csv"
eval_meta_path = QA_DIR / "0_data" / "metadata_eval.csv"

for path in [train_meta_path, eval_meta_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing metadata file: {path}")

# print(train_meta_path)
# print(eval_meta_path)
print("Train metadata:", train_meta_path)
print("Eval metadata:", eval_meta_path)
# =====

metadata_train.csv: 0.00B [00:00, ?B/s]

/root/.cache/huggingface/hub/datasets--KickItLikeShika--NileTTS-dataset/snapshots/6905cb892bda2a315377178729ab2e718c0e63d4/metadata_train.csv
/root/.cache/huggingface/hub/datasets--KickItLikeShika--NileTTS-dataset/snapshots/6905cb892bda2a315377178729ab2e718c0e63d4/metadata_eval.csv


In [6]:
def load_niletts_metadata(path):
    df = pd.read_csv(
        path,
        sep="|",
        names=["audio_file", "text", "speaker_name"],
        dtype=str,
        engine="python",
        quoting=csv.QUOTE_NONE,
        on_bad_lines="skip"
    )

    if len(df) > 0 and str(df.iloc[0]["audio_file"]).strip() == "audio_file":
        df = df.iloc[1:].copy()

    for col in ["audio_file", "text", "speaker_name"]:
        df[col] = df[col].astype(str).str.strip()

    df = df.dropna(subset=["audio_file", "text", "speaker_name"])
    df = df[df["text"].str.len() > 0]
    df = df.drop_duplicates(subset=["audio_file"]).reset_index(drop=True)
    return df


def extract_domain(audio_file):
    name = os.path.basename(str(audio_file)).lower()
    if name.startswith("sales"):
        return "sales"
    if name.startswith("medical"):
        return "medical"
    if name.startswith("general"):
        return "general"
    return "unknown"


train_df = load_niletts_metadata(train_meta_path)
eval_df = load_niletts_metadata(eval_meta_path)

train_df["domain"] = train_df["audio_file"].apply(extract_domain)
eval_df["domain"] = eval_df["audio_file"].apply(extract_domain)

print("Train metadata:", train_df.shape)
print("Eval metadata:", eval_df.shape)

display(train_df.head())
display(eval_df.head())


Train metadata: (8571, 4)
Eval metadata: (950, 4)


,audio_file,text,speaker_name,domain
0,wavs/general_1_00041.wav,تماما وبجانب درجة الجودة العامة بيسألهم أسئلة ...,SPEAKER_02,general
1,wavs/medical_8_00009.wav,لأن العواقب أسوأ بكتير من اللي بنتخيله بص، الل...,SPEAKER_02,medical
2,wavs/sales_15_00036.wav,بنبص بره على العميل و القيمة اللي بيحصل عليها....,SPEAKER_01,sales
3,wavs/sales_3_00038.wav,تاني حاجة خططك كلها بتتدمر. حلم إنك تستقر في ب...,SPEAKER_02,sales
4,wavs/sales_26_00088.wav,هنا الأمور بدأت تتغير بشكل ملموس قوي، الإنترنت...,SPEAKER_02,sales


,audio_file,text,speaker_name,domain
0,wavs/sales_34_00019.wav,فالشخص اللي هو يعني حافز مش فاهم ده. اللي قافل...,SPEAKER_02,sales
1,wavs/sales_26_00082.wav,نكون خدنا جولة سريعة و مكثفة يعني في أهم الأفك...,SPEAKER_01,sales
2,wavs/sales_15_00055.wav,لو التجربة كانت سيئة يعني اتعامل وحش. المنتج ط...,SPEAKER_01,sales
3,wavs/medical_10_00012.wav,قاعدة اسمها 20-20-20 بالزبط وبسطتها دي هي سر ق...,SPEAKER_02,medical
4,wavs/general_6_00008.wav,الطالب هنا فعليا اتحول من متلقي للعلم لمجرد رق...,SPEAKER_01,general


In [7]:
print("Train domains")
display(train_df["domain"].value_counts())

print("Eval domains")
display(eval_df["domain"].value_counts())

print("Train speakers")
display(train_df["speaker_name"].value_counts())

print("Eval speakers")
display(eval_df["speaker_name"].value_counts())


Train domains


,count
domain,
sales,4478
general,2682
medical,1411


Eval domains


,count
domain,
sales,497
general,297
medical,156


Train speakers


,count
speaker_name,
SPEAKER_01,4385
SPEAKER_02,4186


Eval speakers


,count
speaker_name,
SPEAKER_01,480
SPEAKER_02,470


## 2. Split train metadata into QA-train and QA-validation

The official eval metadata stays untouched as the final QA test source.


In [8]:
train_df["strata"] = train_df["domain"] + "_" + train_df["speaker_name"]

train_source, val_source = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_df["strata"]
)

test_source = eval_df.copy()

train_source = train_source.reset_index(drop=True)
val_source = val_source.reset_index(drop=True)
test_source = test_source.reset_index(drop=True)

print("QA train source rows:", len(train_source))
print("QA val source rows:", len(val_source))
print("QA test source rows:", len(test_source))


QA train source rows: 7713
QA val source rows: 858
QA test source rows: 950


## 3. LLM setup for QA pair generation

This notebook uses Gemini because it can generate structured JSON.

In Colab, add your key using:

`Secrets → Add new secret → GEMINI_API_KEY`

Then enable notebook access to the secret.


In [ ]:
# from google import genai
# from pydantic import BaseModel, Field
# from typing import List

# try:
#     from google.colab import userdata
#     GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
# except Exception:
#     GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

# if not GEMINI_API_KEY:
#     raise ValueError("Missing GEMINI_API_KEY. Add it in Colab Secrets or set it as an environment variable.")

# client = genai.Client(api_key=GEMINI_API_KEY)

# GEMINI_MODEL = "gemini-2.5-flash"


In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

LOCAL_LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM_NAME)

local_llm = AutoModelForCausalLM.from_pretrained(
    LOCAL_LLM_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

local_llm.eval()


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

## 4. Structured schema for generated QA pairs

The LLM must return JSON only. Each answer must be copied exactly from the transcript so that we can use it as a gold answer for QA training/evaluation.


In [10]:
from pydantic import BaseModel, Field
from typing import List


class GeneratedQA(BaseModel):
    question: str = Field(description="A clear Arabic question based only on the transcript.")
    answer: str = Field(description="The exact answer copied from the transcript.")
    question_type: str = Field(description="One of: main_idea, detail_extraction, number_extraction, intent, domain_specific")
    difficulty: str = Field(description="One of: easy, medium")


class GeneratedQAList(BaseModel):
    qa_pairs: List[GeneratedQA]


In [11]:
def normalize_spaces(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def find_answer_start_soft(context, answer):
    context_norm = normalize_spaces(context)
    answer_norm = normalize_spaces(answer)

    exact = context.find(answer)
    if exact != -1:
        return exact

    norm_pos = context_norm.find(answer_norm)
    if norm_pos != -1:
        return norm_pos

    return -1


def build_qa_generation_prompt(context, domain, speaker_name, n_pairs=3):
    return f'''
You are creating a high-quality Arabic question-answering dataset from an Egyptian Arabic transcript.

Transcript domain: {domain}
Speaker: {speaker_name}

Transcript:
"""{context}"""

Generate {n_pairs} useful QA pairs.

Rules:
1. Questions must be in Arabic.
2. Questions must be answerable ONLY from the transcript.
3. The answer must be a short span copied EXACTLY from the transcript.
4. Do not write answers that are not literally present in the transcript.
5. Avoid yes/no questions.
6. Avoid vague questions such as "what is mentioned?"
7. Use varied question types:
   - main_idea
   - detail_extraction
   - intent
   - domain_specific
   - number_extraction only if there is a number in the transcript
8. For medical transcripts, ask about symptoms, advice, treatment, diagnosis, or health information if available.
9. For sales transcripts, ask about customer need, product/service, price, offer, request, or buying/selling information if available.
10. For general transcripts, ask about the main idea, action, reason, or important detail.
11. Return JSON only using the required schema.
'''


import json
import re
import time

# from google.genai import types


# def safe_json_loads(text):
#     text = str(text).strip()

#     if text.startswith("```json"):
#         text = text.replace("```json", "").replace("```", "").strip()
#     elif text.startswith("```"):
#         text = text.replace("```", "").strip()

#     return json.loads(text)


# def call_llm_generate_qa(context, domain, speaker_name, n_pairs=3, max_retries=3):
#     prompt = build_qa_generation_prompt(
#         context=context,
#         domain=domain,
#         speaker_name=speaker_name,
#         n_pairs=n_pairs
#     )

#     last_error = None

#     for attempt in range(max_retries):
#         try:
#             response = client.models.generate_content(
#                 model=GEMINI_MODEL,
#                 contents=prompt,
#                 config=types.GenerateContentConfig(
#                     temperature=0.2,
#                     response_mime_type="application/json",
#                     response_schema=GeneratedQAList
#                 )
#             )

#             parsed = safe_json_loads(response.text)

#             if isinstance(parsed, dict) and "qa_pairs" in parsed:
#                 return parsed["qa_pairs"]

#             return []

#         except Exception as e:
#             last_error = e
#             wait_time = 2 ** attempt
#             print(f"LLM error, retrying in {wait_time}s: {e}")
#             time.sleep(wait_time)

#     print("Failed after retries:", last_error)
#     return None

def extract_json_from_text(text):
    text = str(text).strip()

    if "```json" in text:
        text = text.split("```json", 1)[1].split("```", 1)[0].strip()
    elif "```" in text:
        text = text.split("```", 1)[1].split("```", 1)[0].strip()

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        text = text[start:end+1]

    return json.loads(text)


def call_llm_generate_qa(context, domain, speaker_name, n_pairs=3, max_retries=2):
    prompt = build_qa_generation_prompt(
        context=context,
        domain=domain,
        speaker_name=speaker_name,
        n_pairs=n_pairs
    )

    system_prompt = """
You are an Arabic NLP dataset creator.
Return JSON only.
Do not return explanations.
The answer must be copied exactly from the transcript.
"""

    user_prompt = f"""
{prompt}

Return exactly this JSON structure:
{{
  "qa_pairs": [
    {{
      "question": "Arabic question",
      "answer": "exact answer span copied from transcript",
      "question_type": "main_idea/detail_extraction/intent/domain_specific/number_extraction",
      "difficulty": "easy/medium"
    }}
  ]
}}
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(local_llm.device)

    last_error = None

    for attempt in range(max_retries):
        try:
            with torch.no_grad():
                outputs = local_llm.generate(
                    **inputs,
                    max_new_tokens=512,
                    do_sample=True,
                    temperature=0.2,
                    top_p=0.9,
                    pad_token_id=llm_tokenizer.eos_token_id
                )

            generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
            generated_text = llm_tokenizer.decode(
                generated_ids,
                skip_special_tokens=True
            )

            parsed = extract_json_from_text(generated_text)

            if isinstance(parsed, dict) and "qa_pairs" in parsed:
                return parsed["qa_pairs"]

            return []

        except Exception as e:
            last_error = e
            print(f"Local LLM parsing error, retrying: {e}")
            time.sleep(1)

    print("Failed local generation:", last_error)
    return []


## 5. Generate LLM QA pairs

In [ ]:
def generate_llm_qa_dataframe_checkpointed(
    source_df,
    split_name,
    checkpoint_path,
    processed_path,
    max_rows=None,
    n_pairs_per_transcript=3,
    save_every=10,
    sleep_seconds=0
):
    df = source_df.copy().reset_index(drop=True)

    if max_rows is not None:
        df = df.head(max_rows)

    checkpoint_path = Path(checkpoint_path)
    processed_path = Path(processed_path)

    if checkpoint_path.exists():
        all_rows = pd.read_csv(checkpoint_path).to_dict("records")
        print(f"Loaded existing checkpoint: {checkpoint_path} with {len(all_rows)} QA rows")
    else:
        all_rows = []

    if processed_path.exists():
        processed_df = pd.read_csv(processed_path)
        processed_indices = set(processed_df["source_row_index"].astype(int).tolist())
        print(f"Loaded processed log: {len(processed_indices)} transcript rows already processed")
    else:
    # ======
        # processed_indices = set()
        processed_indices = {
            int(row["source_row_index"])
            for row in all_rows
        }
        print(f"No processed log found; recovered {len(processed_indices)} source rows from the checkpoint")
    # =====

    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"LLM QA generation: {split_name}"):
        if i in processed_indices:
            continue

        context = str(row["text"]).strip()
        domain = row["domain"]
        speaker_name = row["speaker_name"]
        audio_file = row["audio_file"]

        qa_pairs = call_llm_generate_qa(
            context=context,
            domain=domain,
            speaker_name=speaker_name,
            n_pairs=n_pairs_per_transcript
        )

        # Important: if the local LLM fails/parses badly and returns [] or None,
        # do NOT mark the transcript as processed. It can be retried in the next run.
        if not qa_pairs:
            print(f"No valid generation for row {i}. It will be retried next run.")
            continue

        added_for_this_row = 0

        for j, qa in enumerate(qa_pairs):
            if not isinstance(qa, dict):
                continue

            answer = normalize_spaces(qa.get("answer", ""))
            question = normalize_spaces(qa.get("question", ""))
            answer_start = find_answer_start_soft(context, answer)

            if not question or not answer:
                continue

            all_rows.append({
                "id": f"{split_name}_{i}_{j}",
                "split": split_name,
                "source_row_index": i,
                "audio_file": audio_file,
                "speaker_name": speaker_name,
                "domain": domain,
                "context_ref": context,
                "question": question,
                "answer": answer,
                "answer_start": answer_start,
                "question_type": qa.get("question_type", ""),
                "difficulty": qa.get("difficulty", ""),
                "valid_answer_span": answer_start != -1,
                "needs_manual_review": True
            })

            added_for_this_row += 1

        if added_for_this_row == 0:
            print(f"Generated output for row {i}, but no usable QA rows. It will be retried next run.")
            continue

        processed_indices.add(i)

        if len(processed_indices) % save_every == 0:
            pd.DataFrame(all_rows).to_csv(
                checkpoint_path,
                index=False,
                encoding="utf-8-sig"
            )

            pd.DataFrame({
                "source_row_index": sorted(list(processed_indices))
            }).to_csv(
                processed_path,
                index=False,
                encoding="utf-8-sig"
            )

            print(f"Saved checkpoint for {split_name}: {len(processed_indices)} transcript rows processed")

        time.sleep(sleep_seconds)

    pd.DataFrame(all_rows).to_csv(
        checkpoint_path,
        index=False,
        encoding="utf-8-sig"
    )

    pd.DataFrame({
        "source_row_index": sorted(list(processed_indices))
    }).to_csv(
        processed_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Finished {split_name}: {len(processed_indices)} transcript rows processed")
    return pd.DataFrame(all_rows)


In [ ]:
MAX_TRAIN_ROWS = None
MAX_VAL_ROWS = None
MAX_TEST_ROWS = None

N_PAIRS_PER_TRANSCRIPT = 3
SAVE_EVERY = 10
SLEEP_SECONDS = 0

# qa_train_df = generate_llm_qa_dataframe_checkpointed(
#     train_source,
#     split_name="train",
#     checkpoint_path="checkpoint_llm_qa_train.csv",
#     processed_path="processed_llm_qa_train.csv",
#     max_rows=MAX_TRAIN_ROWS,
#     n_pairs_per_transcript=N_PAIRS_PER_TRANSCRIPT,
#     save_every=SAVE_EVERY,
#     sleep_seconds=SLEEP_SECONDS
# )

qa_val_df = generate_llm_qa_dataframe_checkpointed(
    val_source,
    split_name="val",
    # =====
    # checkpoint_path="checkpoint_llm_qa_val.csv",
    # processed_path="processed_llm_qa_val.csv",
    checkpoint_path=CHECKPOINT_VAL_FILE,
    processed_path=PROCESSED_VAL_FILE,
    # =====
    max_rows=MAX_VAL_ROWS,
    n_pairs_per_transcript=N_PAIRS_PER_TRANSCRIPT,
    save_every=SAVE_EVERY,
    sleep_seconds=SLEEP_SECONDS
)

# qa_test_df = generate_llm_qa_dataframe_checkpointed(
#     test_source,
#     split_name="test",
#     checkpoint_path="checkpoint_llm_qa_test.csv",
#     processed_path="processed_llm_qa_test.csv",
#     max_rows=MAX_TEST_ROWS,
#     n_pairs_per_transcript=N_PAIRS_PER_TRANSCRIPT,
#     save_every=SAVE_EVERY,
#     sleep_seconds=SLEEP_SECONDS
# )

# print("Raw LLM QA train:", qa_train_df.shape)
print("Raw LLM QA val:", qa_val_df.shape)
# print("Raw LLM QA test:", qa_test_df.shape)

# display(qa_train_df.head())
display(qa_val_df.head())
# display(qa_test_df.head())


LLM QA generation: val:   0%|          | 0/858 [00:00<?, ?it/s]

Saved checkpoint for val: 10 transcript rows processed
Saved checkpoint for val: 20 transcript rows processed
Saved checkpoint for val: 30 transcript rows processed
Saved checkpoint for val: 40 transcript rows processed
Saved checkpoint for val: 50 transcript rows processed
Saved checkpoint for val: 60 transcript rows processed
Saved checkpoint for val: 70 transcript rows processed
Saved checkpoint for val: 80 transcript rows processed
Saved checkpoint for val: 90 transcript rows processed
Saved checkpoint for val: 100 transcript rows processed
Saved checkpoint for val: 110 transcript rows processed
Saved checkpoint for val: 120 transcript rows processed
Saved checkpoint for val: 130 transcript rows processed
Saved checkpoint for val: 140 transcript rows processed
Saved checkpoint for val: 150 transcript rows processed
Saved checkpoint for val: 160 transcript rows processed
Saved checkpoint for val: 170 transcript rows processed
Saved checkpoint for val: 180 transcript rows processed
S

In [1]:
!find /content -type f | grep -E "checkpoint|processed|niletts|qa|val|arat5"

### delete the bad checkpoint files if they exist

In [ ]:
# from pathlib import Path
# import pandas as pd

# for f in [
#     "checkpoint_llm_qa_test.csv",
#     "processed_llm_qa_test.csv"
# ]:
#     p = Path(f)
#     print(f, "exists:", p.exists())
#     if p.exists():
#         df = pd.read_csv(p)
#         print("shape:", df.shape)
#         display(df.tail())

checkpoint_llm_qa_test.csv exists: False
processed_llm_qa_test.csv exists: False


In [ ]:
# from pathlib import Path

# for f in [
#     "checkpoint_llm_qa_test.csv",
#     "processed_llm_qa_test.csv"
# ]:
#     p = Path(f)
#     if p.exists():
#         p.unlink()
#         print("Deleted:", f)

### Use this cell after you finish generating test, val, and train.
### It reloads the generated checkpoint CSVs into qa_train_df, qa_val_df, qa_test_df.


In [ ]:
# =====
# import pandas as pd
# from pathlib import Path

# def load_checkpoint_csv(path):
def load_qa_csv(path):
    path = Path(path)
    # if not path.exists():
    #     raise FileNotFoundError(f"Missing file: {path}. Generate this split first.")
    if not path.is_file():
        raise FileNotFoundError(f"Missing QA input file: {path}")

    df = pd.read_csv(path)
    print(path.name, df.shape)
    return df

# # Uncomment these only after all three checkpoint files exist:
# # qa_train_df = load_checkpoint_csv("checkpoint_llm_qa_train.csv")
# # qa_val_df = load_checkpoint_csv("checkpoint_llm_qa_val.csv")
# # qa_test_df = load_checkpoint_csv("checkpoint_llm_qa_test.csv")

qa_train_df = load_qa_csv(TRAIN_FILE)
qa_val_df = load_qa_csv(CHECKPOINT_TEST_FILE)

# =====


checkpoint_llm_qa_test.csv (2818, 14)


## 6. Filter invalid generated pairs

We keep only QA pairs where the answer exists in the reference transcript.

This is very important because the QA model needs a reliable gold answer.

### Run this cell ONLY after qa_train_df, qa_val_df, and qa_test_df all exist.


In [ ]:
def filter_valid_qa(df):
    before = len(df)
    df = df.copy()

    df["question"] = df["question"].astype(str).map(normalize_spaces)
    df["answer"] = df["answer"].astype(str).map(normalize_spaces)
    df["context_ref"] = df["context_ref"].astype(str).map(normalize_spaces)

    df = df[df["question"].str.len() > 5]
    df = df[df["answer"].str.len() > 0]

    # ======
    # df = df[df["valid_answer_span"] == True]
    span_column = (
        "valid_answer_span"
        if "valid_answer_span" in df.columns
        else "valid_answer_span_ref"
    )

    df = df[df[span_column] == True]
    # ======

    df = df.drop_duplicates(subset=["audio_file", "question", "answer"]).reset_index(drop=True)

    print(f"Kept {len(df)} / {before} rows")
    return df

# =====
# Run this ONLY after qa_train_df, qa_val_df, and qa_test_df all exist.
qa_train_df = filter_valid_qa(qa_train_df)
qa_val_df = filter_valid_qa(qa_val_df)
qa_test_df = filter_valid_qa(qa_test_df)

# qa_train_df.to_csv("niletts_llm_qa_train_generated.csv", index=False, encoding="utf-8-sig")
qa_train_df.to_csv(QA_DIR / "niletts_llm_qa_train_generated.csv",index=False,encoding="utf-8-sig")

# qa_val_df.to_csv("niletts_llm_qa_val_generated.csv", index=False, encoding="utf-8-sig")
qa_val_df.to_csv(QA_DIR / "niletts_llm_qa_val_generated.csv",index=False,encoding="utf-8-sig")

# qa_test_df.to_csv("niletts_llm_qa_test_reference_generated.csv", index=False, encoding="utf-8-sig")
qa_test_df.to_csv(QA_DIR / "niletts_llm_qa_test_reference_generated.csv",index=False,encoding="utf-8-sig")


qa_all_df = pd.concat([qa_train_df, qa_val_df, qa_test_df], ignore_index=True)

# qa_all_df.to_csv("niletts_llm_qa_all_generated_for_review.csv", index=False, encoding="utf-8-sig")
qa_all_df.to_csv(QA_DIR / "niletts_llm_qa_all_generated_for_review.csv",index=False,encoding="utf-8-sig")
# =====

print("Final QA train:", qa_train_df.shape)
print("Final QA val:", qa_val_df.shape)
print("Final QA test:", qa_test_df.shape)

display(qa_train_df.sample(min(5, len(qa_train_df)), random_state=SEED))


## 7. Manual review step

Before training AraT5, open these CSV files and review them:

- `niletts_llm_qa_train_generated.csv`
- `niletts_llm_qa_val_generated.csv`
- `niletts_llm_qa_test_reference_generated.csv`

At minimum, manually review the test file because it will be used for final reporting.

After review, save your edited files as:
- `niletts_llm_qa_train_reviewed.csv`
- `niletts_llm_qa_val_reviewed.csv`
- `niletts_llm_qa_test_reference_reviewed.csv`

Then set `USE_REVIEWED_FILES = True`.


In [ ]:
USE_REVIEWED_FILES = False

if USE_REVIEWED_FILES:
    # =====
    # qa_train_df = pd.read_csv("niletts_llm_qa_train_reviewed.csv")
    qa_train_df = pd.read_csv(QA_DIR / "niletts_llm_qa_train_reviewed.csv")

    # qa_val_df = pd.read_csv("niletts_llm_qa_val_reviewed.csv")
    qa_val_df = pd.read_csv(QA_DIR / "niletts_llm_qa_val_reviewed.csv")

    # qa_test_df = pd.read_csv("niletts_llm_qa_test_reference_reviewed.csv")
    qa_test_df = pd.read_csv(QA_DIR / "niletts_llm_qa_test_reference_reviewed.csv")
    # =====

print(qa_train_df.shape, qa_val_df.shape, qa_test_df.shape)


### Free GPU memory before loading AraT5.
### Run this after finishing QA generation and before the AraT5 training cells.


In [ ]:
import gc

try:
    del local_llm
    del llm_tokenizer
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA memory cleared.")


## 8. Prepare AraT5 training data

### AraT5 input format

Input:
```text
أجب عن السؤال اعتمادا على النص.
السؤال: ...
النص: ...
```

Target:
```text
answer
```


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

MODEL_NAME = "UBC-NLP/AraT5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(device)

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 96


def build_t5_input(question, context):
    return f"أجب عن السؤال اعتمادا على النص. السؤال: {question} النص: {context}"


def prepare_for_t5(df):
    out = df.copy()
    out["input_text"] = out.apply(
        lambda r: build_t5_input(r["question"], r["context_ref"]),
        axis=1
    )
    out["target_text"] = out["answer"].astype(str)
    return out


t5_train_df = prepare_for_t5(qa_train_df)
t5_val_df = prepare_for_t5(qa_val_df)
t5_test_df = prepare_for_t5(qa_test_df)

train_dataset = Dataset.from_pandas(t5_train_df[["input_text", "target_text"]])
val_dataset = Dataset.from_pandas(t5_val_df[["input_text", "target_text"]])
test_dataset = Dataset.from_pandas(t5_test_df[["input_text", "target_text"]])


def preprocess_batch(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_dataset.map(preprocess_batch, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess_batch, batched=True, remove_columns=val_dataset.column_names)
tokenized_test = test_dataset.map(preprocess_batch, batched=True, remove_columns=test_dataset.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


## 9. Exact Match and token-F1 metrics

In [ ]:
AR_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
PUNCT = re.compile(r"[^\w\s\u0600-\u06FF]")


def normalize_arabic(text):
    text = str(text)
    text = AR_DIACRITICS.sub("", text)
    text = text.replace("ـ", "")
    text = re.sub(r"[إأآا]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ة", "ه")
    text = PUNCT.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def exact_match_score(prediction, ground_truth):
    return int(normalize_arabic(prediction) == normalize_arabic(ground_truth))


def token_f1_score(prediction, ground_truth):
    pred_tokens = normalize_arabic(prediction).split()
    gold_tokens = normalize_arabic(ground_truth).split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = {}
    for tok in set(pred_tokens):
        common[tok] = min(pred_tokens.count(tok), gold_tokens.count(tok))

    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    em_scores = [exact_match_score(p, g) for p, g in zip(decoded_preds, decoded_labels)]
    f1_scores = [token_f1_score(p, g) for p, g in zip(decoded_preds, decoded_labels)]

    return {
        "exact_match": float(np.mean(em_scores)),
        "f1": float(np.mean(f1_scores))
    }


## 10. Fine-tune AraT5

In [ ]:
training_args = Seq2SeqTrainingArguments(
    # =====
    # output_dir="arat5_niletts_llm_reference_qa",
    output_dir=str(QA_DIR / "arat5_niletts_llm_reference_qa"),
    # =====

    overwrite_output_dir=True,

    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    weight_decay=0.01,

    # =====
    evaluation_strategy="epoch",
    eval_strategy="epoch",
    # =====

    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,

    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
# =====
# trainer.save_model("arat5_niletts_llm_reference_qa_best")
# tokenizer.save_pretrained("arat5_niletts_llm_reference_qa_best")

best_model_dir = str(
    QA_DIR / "arat5_niletts_llm_reference_qa_best"
)

trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)
# =====


## 11. Evaluate AraT5 on reference transcripts

In [ ]:
pred_output = trainer.predict(tokenized_test)

pred_ids = pred_output.predictions
label_ids = pred_output.label_ids

label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
gold_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

eval_predictions_df = t5_test_df.copy()
eval_predictions_df["pred_answer_ref_context"] = pred_texts
eval_predictions_df["gold_answer"] = gold_texts

eval_predictions_df["em_ref"] = [
    exact_match_score(p, g)
    for p, g in zip(pred_texts, gold_texts)
]

eval_predictions_df["f1_ref"] = [
    token_f1_score(p, g)
    for p, g in zip(pred_texts, gold_texts)
]

print("Reference Transcript QA Exact Match:", eval_predictions_df["em_ref"].mean())
print("Reference Transcript QA F1:", eval_predictions_df["f1_ref"].mean())

# =====
# eval_predictions_df.to_csv(
#     "niletts_arat5_llm_reference_eval_predictions.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

eval_predictions_df.to_csv(
    QA_DIR / "niletts_arat5_llm_reference_eval_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)
# =====

display(eval_predictions_df[[
    "audio_file", "domain", "question", "context_ref",
    "gold_answer", "pred_answer_ref_context", "em_ref", "f1_ref"
]].head())


## 12. Later: evaluate the same QA questions on ASR transcripts

Run this only after your teammate provides `niletts_asr_eval_results.csv`.

Expected columns:
- `audio_file`
- `asr_text`
- `wer`
- `cer`


In [ ]:
# # =====
# # asr_eval_df = pd.read_csv("niletts_asr_eval_results.csv")
# asr_eval_df = pd.read_csv(ASR_FILE)
# # =====

# qa_test_with_asr = qa_test_df.merge(
#     asr_eval_df[["audio_file", "asr_text", "wer", "cer"]],
#     on="audio_file",
#     how="left"
# ).rename(columns={
#     "asr_text": "context_asr",
#     "wer": "asr_wer",
#     "cer": "asr_cer"
# })

# qa_test_with_asr["input_text"] = qa_test_with_asr.apply(
#     lambda r: build_t5_input(r["question"], r["context_asr"]),
#     axis=1
# )

# asr_dataset = Dataset.from_pandas(
#     qa_test_with_asr[["input_text", "answer"]].rename(columns={"answer": "target_text"})
# )

# tokenized_asr_test = asr_dataset.map(preprocess_batch, batched=True, remove_columns=asr_dataset.column_names)

# asr_pred_output = trainer.predict(tokenized_asr_test)
# asr_pred_texts = tokenizer.batch_decode(asr_pred_output.predictions, skip_special_tokens=True)

# qa_test_with_asr["pred_answer_asr_context"] = asr_pred_texts
# qa_test_with_asr["em_asr"] = [
#     exact_match_score(p, g)
#     for p, g in zip(asr_pred_texts, qa_test_with_asr["answer"])
# ]
# qa_test_with_asr["f1_asr"] = [
#     token_f1_score(p, g)
#     for p, g in zip(asr_pred_texts, qa_test_with_asr["answer"])
# ]

# ref_em = eval_predictions_df["em_ref"].mean()
# ref_f1 = eval_predictions_df["f1_ref"].mean()
# asr_em = qa_test_with_asr["em_asr"].mean()
# asr_f1 = qa_test_with_asr["f1_asr"].mean()

# print("Reference QA EM:", ref_em)
# print("Reference QA F1:", ref_f1)
# print("ASR QA EM:", asr_em)
# print("ASR QA F1:", asr_f1)
# print("EM drop caused by ASR:", ref_em - asr_em)
# print("F1 drop caused by ASR:", ref_f1 - asr_f1)

# # =====
# # qa_test_with_asr.to_csv(
# #     "niletts_arat5_llm_asr_eval_predictions.csv",
# #     index=False,
# #     encoding="utf-8-sig"
# # )

# qa_test_with_asr.to_csv(
# QA_DIR / "niletts_arat5_llm_asr_eval_predictions.csv",
#     index=False,
#     encoding="utf-8-sig"
# )
# # =====

# display(qa_test_with_asr.head())
